# Phase 1, test P1a: is the Urban100 tile gain a hidden ensemble? (CPU only)

`urban100-tiles` blended 512-px tiles (overlap 128) with linear ramps, so pixels in an overlap average 2–4 tile
estimates per pass; the ×2 flip runs the tiling on the mirrored image, so its grid is anchored at the other edge.
On pixels that get exactly **one** tile estimate in both passes ("single"), tiles ×2 and whole ×2 each average exactly
two estimates, so any gain there comes from the restricted context / resolution, not from averaging.

Pre-registered (decision log, Phase 1): on confirm-88, the resolution account survives if at σ 25 and σ 50 the mean
single-pixel gain is ≥ 0.5 × the mean all-pixel gain and the single-pixel gain is > 0 with sign-flip p < 0.01.
Gains are 10·log10(MSE_whole / MSE_tiles) per image and region, on the uint8-rounded ×2 pilots.
Kaggle: CPU, internet on, input = kernel output `trnquanghuyn/urban100-tiles`.

In [ ]:
import glob, io, json, math, os, re, urllib.request, zipfile
import numpy as np
import pandas as pd
from PIL import Image

SIGMAS = [15, 25, 50]
DEV12 = list(range(8, 100, 8))
TILE, OVERLAP = 512, 128
OUT, CACHE = "outputs", "/tmp/cache"
for d in (OUT, CACHE):
    os.makedirs(d, exist_ok=True)

ROOT = sorted(glob.glob("/kaggle/input/**/pilots", recursive=True))
assert ROOT, "attach the output of trnquanghuyn/urban100-tiles"
ROOT = os.path.dirname(ROOT[0])
G1 = pd.read_csv(f"{ROOT}/scores.csv")

path = f"{CACHE}/Datasets.zip"
if not os.path.exists(path):
    urllib.request.urlretrieve("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", path)


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


with zipfile.ZipFile(path) as z:
    names = sorted([n for n in z.namelist() if n.startswith("Datasets/Gaussion/Urban100/") and not n.endswith("/")], key=natkey)
    TEST = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))) for n in names]
print(ROOT, len(TEST), "Urban100 images")

## Coverage of `nbU2.tiled` (same padding rule, stride and tile origins)

In [ ]:
def coverage(H, W):
    stride = TILE - OVERLAP
    cover = lambda n: max(TILE, TILE + math.ceil(max(n - TILE, 0) / stride) * stride)
    Hp, Wp = cover(H), cover(W)
    c = np.zeros((Hp, Wp), np.int16)
    for i in range(0, Hp - TILE + 1, stride):
        for j in range(0, Wp - TILE + 1, stride):
            c[i:i + TILE, j:j + TILE] += 1
    return c[:H, :W]


def load(kind, sigma, idx):
    x = np.load(f"{ROOT}/pilots/{kind}_s{sigma}_{idx:03d}.npz")["x"].astype(np.float64).transpose(1, 2, 0)
    return np.round(np.clip(x, 0, 1) * 255)


def region_gain(ew, et, m):
    return 10 * math.log10(ew[m].mean() / et[m].mean())


rows = []
for sigma in SIGMAS:
    for idx, (name, clean_u8) in enumerate(TEST):
        clean = clean_u8.astype(np.float64)
        H, W = clean.shape[:2]
        c = coverage(H, W)
        single = (c == 1) & (np.fliplr(c) == 1)
        ew = ((load("whole", sigma, idx) - clean) ** 2).mean(-1)
        et = ((load("tiles-512", sigma, idx) - clean) ** 2).mean(-1)
        full = np.ones_like(single)
        rows.append(dict(idx=idx, sigma=sigma, split="dev" if idx in DEV12 else "confirm", frac_single=float(single.mean()),
                         gain_all=region_gain(ew, et, full), gain_single=region_gain(ew, et, single),
                         gain_multi=region_gain(ew, et, ~single)))
R = pd.DataFrame(rows)
R.to_csv(f"{OUT}/regions.csv", index=False)

# Consistency with the GPU run's per-image scores (float16 pilots: expect ≲ 0.001 dB)
g = G1[G1.method.isin(["tiles-512 ×2", "whole ×2"])].pivot_table(index=["idx", "sigma"], columns="method", values="psnr")
chk = R.set_index(["idx", "sigma"]).gain_all - (g["tiles-512 ×2"] - g["whole ×2"])
print(f"gain_all vs GPU scores: max |Δ| = {chk.abs().max():.4f} dB")

## Pre-registered test (confirm-88)

In [ ]:
def sign_flip_p(d, n_flip=200_000, seed=0):
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1.0, 1.0], size=(n_flip, len(d)))
    return float((np.abs((signs * d).mean(1)) >= abs(d.mean()) - 1e-12).mean())


out = []
for sigma, g in R[R.split == "confirm"].groupby("sigma"):
    out.append(dict(sigma=sigma, n=len(g), frac_single=g.frac_single.mean(), gain_all=g.gain_all.mean(),
                    gain_single=g.gain_single.mean(), gain_multi=g.gain_multi.mean(),
                    share=g.gain_single.mean() / g.gain_all.mean(), single_better=int((g.gain_single > 0).sum()),
                    p_single=sign_flip_p(g.gain_single.to_numpy())))
T = pd.DataFrame(out)
print(T.round(4).to_string(index=False))
survives = bool(all((T.set_index("sigma").loc[s, "share"] >= 0.5) and (T.set_index("sigma").loc[s, "p_single"] < 0.01)
                    and (T.set_index("sigma").loc[s, "gain_single"] > 0) for s in (25, 50)))
print("resolution account survives (σ 25 and 50):", survives)
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(tests=T.to_dict("records"), survives=survives, max_consistency_gap=float(chk.abs().max())), f, indent=1, default=float)